In [3]:
from regina import Triangulation3, Perm4
import random
import numpy as np

In [4]:
# functions

In [5]:
def gen_acceptable(size):
    tri = Triangulation3()
    for _ in range(size):
        tri.newTetrahedron()

    open_faces = []
    for i in range(size):
        tet = tri.tetrahedron(i)
        for f in range(4):
            open_faces.append((tet, f))

    random.shuffle(open_faces)

    while open_faces:
        tet1, face1 = open_faces.pop()
        tet2, face2 = open_faces.pop()

        perm_list = [0, 0, 0, 0]
        perm_list[face1] = face2

        rem_verts1 = [v for v in range(4) if v != face1]
        rem_verts2 = [v for v in range(4) if v != face2]

        random.shuffle(rem_verts2)

        for i in range(3):
            perm_list[rem_verts1[i]] = rem_verts2[i]

        gluing = Perm4(perm_list[0], perm_list[1], perm_list[2], perm_list[3])
        tet1.join(face1, tet2, gluing)

    return tri

def isManifold(tri):
    is_manifold = True
    for i in range(tri.countVertices()):
        v = tri.vertex(i)
        if v.linkEulerChar() != 2:
            is_manifold = False
            break

    return is_manifold

In [103]:
# generate random strings

In [109]:
N = 100_000

size = 10
sigs = []
sig = ""

for _ in range(N):
    tri = gen_acceptable(size)
    if tri.isConnected():
        sig = tri.dehydrate()
        sigs.append(sig)

av_chars = {}
seq_len = len(sig)

for cid in range(seq_len):
    av_chars[cid] = list(set([sig[cid] for sig in sigs]))

alpha_ods = np.prod([len(av_chars[i]) / 26 for i in av_chars])
print(f"{size}: {alpha_ods}")

10: 1.8813173381618386e-12


In [ ]:
acceptable_count = 0
valid_count = 0
N = 100_000_000

for _ in range(N):
    word = "".join([random.choice(av_chars[cid]) for cid in range(seq_len)])

    try:
        t = Triangulation3.rehydrate(word)
        valid_count += 1 if t.isValid() else 0
        acceptable_count += 1
    except Exception:
        pass

p = acceptable_count / N
ci = np.sqrt(p * (1 - p) / N)

print(f"{p:.8%} +- {2 * ci:.8%}")

0.00000000% +- 0.00000000%


In [97]:
# compute valid, and closed percentages

In [ ]:
N = 1_000_000_000

for size in [10]:
    connected_count = 0
    valid_count = 0
    closed_count = 0
    sphere_count = 0
    manifold_count = 0

    for _ in range(N):
        tri = gen_acceptable(size)
        if tri.isConnected():
            connected_count += 1
            if tri.isValid():
                valid_count += 1
                if isManifold(tri):
                    manifold_count += 1
                    closed_count += 1 if tri.isClosed() else 0
                    sphere_count += 1 if tri.isSphere() else 0

    print(size, valid_count, connected_count, closed_count)

1000000 204477 940952 116
1000000 195762 952099 10
1000000 188682 960988 1
1000000 182520 967036 0
1000000 177303 971607 0


In [ ]:
116 / 204477

0.0005673009678350133

In [ ]:
10 / 195762

5.1082436836566855e-05

In [ ]:
1/ 188682

5.299922621129731e-06

In [ ]:
# compute sphere percentages